# 09 - Train Online with GRPO (branched rollouts)

This notebook puts **Group Relative Policy Optimization** on paper in Mouse Core terms.

GRPO is PPO's clipped policy update **without a critic**. Instead of GAE from a value head, advantages come from comparing **G stochastic completions** that share the same start:

1. Grow a **trunk** trajectory (env + context of length `L`).
2. At chosen context lengths, **fork**: `deepcopy(env)` × G and copy `context[:L]`.
3. From each fork, sample a different completion with the stochastic policy.
4. Score the G completions (here: sum of rewards on the branch suffix).
5. Z-score those scores inside the group → one advantage per branch.
6. Stamp advantages onto completion steps, train with `GrpoObjective`.

```text
trunk context length L
        │
        ├─ branch 0  ~~~stochastic~~~→  return r0
        ├─ branch 1  ~~~stochastic~~~→  return r1
        ├─ ...
        └─ branch G-1 ~~~~~~~~~~~~~~~→  return r_{G-1}

        A_i = (r_i - mean(r)) / std(r)     # group_relative_advantages
```

Forks happen at **many** `L` values (not only task start), so the policy is trained under short and long in-context histories.

Compared to `08_train_online_ppo.ipynb`: policy head only, no value head; collection is branched rather than a single on-policy stream.


In [ ]:
import copy
from collections import deque

import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_env

from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead
from mouse_core.objectives import (
    GrpoObjective,
    batch_field,
    group_relative_advantages,
    sample_discrete_action,
)


MODEL_ID = "mouse-example-model-grpo"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
MAX_EPISODE_STEPS = 30
EPISODES_PER_TASK = 20

# Small trunk count keeps forking readable; scale up later like the PPO notebook.
NUM_TRUNK_ENVS = 2
GROUP_SIZE = 4              # G completions per fork
BRANCH_EVERY = 32           # fork whenever trunk context length is a multiple of this
BRANCH_HORIZON = 48         # max new steps per branch (also stops on task boundary)
MAX_CONTEXT = 256           # truncate trunk / branch context fed to the model

GRADIENT_STEPS = 2000       # smaller demo budget than full PPO/DQN runs
SEQUENCE_LENGTH = 128
BATCH_SIZE = 4
GRPO_EPOCHS = 2
UPDATES_PER_EPOCH = 25

TRUNK_STEPS_PER_CYCLE = 128  # grow trunks this many steps between train phases


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Environment

Each trunk is a **single** `mouse_gym` env (not a big `GroupEnv`). Forking uses `copy.deepcopy(env)` so every branch starts from the same physical state as the trunk at length `L`, then diverges through stochastic actions.


In [ ]:
def make_trunk_env(i: int):
    return make_env(
        EnvConfig(
            id="Procedural-FrozenLake-v1",
            name=f"proc_frozenlake_grpo_{i}",
            reset_seed=i,
            episodes_per_task=EPISODES_PER_TASK,
            task_reset_options={"regenerate_map": True},
            kwargs={
                "width": 8,
                "height": 8,
                "max_episode_steps": MAX_EPISODE_STEPS,
                "map_seed": i,
                "slippery_success_rate": 1.0,
                "permute_obs": True,
                "permute_actions": True,
            },
        )
    )


trunk_envs = [make_trunk_env(i) for i in range(NUM_TRUNK_ENVS)]


## Model

Policy only: `DiscreteActionHead` → `predictions["action"]`. No value head — GRPO's baseline is the group mean reward.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {"field": "action", "type": "discrete", "vocab_size": MAX_ACTIONS, "std": 0.02, "tokens": 1},
        {"field": "observation", "type": "discrete", "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"field": "reward", "type": "rff", "std": 0.02, "in_min": 0.01, "in_max": 100.0, "tokens": 1},
        {"field": "done", "type": "discrete", "vocab_size": 5, "std": 0.02, "tokens": 1},
    ],
    modality_fusion="sum",
    include_type_token=False,
)

policy_head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=policy_head,  # single head → action_head="action"
).train().to(device)
model.backbone.model.compile()
print(model)


## Branching helpers

- `act_once` — stochastic action + behavior log-prob from the current context.
- `rollout_branch` — from a forked env + shared prefix, sample up to `BRANCH_HORIZON` steps (or until task done codes 3/4).
- `fork_group` — deepcopy env × G, copy context, roll G completions, return rows + scalar returns.

Shared **prefix** rows get `advantage = 0` at stamp time; **suffix** rows get the group-relative `A_i`. Training still feeds prefix+suffix so the transformer sees the same conditioning as at fork time.


In [ ]:
def _truncate(rows: list[dict], max_len: int) -> list[dict]:
    return rows[-max_len:] if len(rows) > max_len else list(rows)


@torch.no_grad()
def act_once(model: Model, context: list[dict]) -> tuple[dict, float]:
    """Sample one action from context; return (input_dict, old_log_prob)."""
    if not context:
        raise ValueError("act_once requires a non-empty context.")
    pending = [_truncate(context, MAX_CONTEXT)]
    predictions, _, _ = model(pending, use_cache=False)
    logits = predictions["action"][:, -1]
    actions, log_probs = sample_discrete_action(logits, num_actions=MAX_ACTIONS)
    return {"action": actions[0].cpu().numpy()}, float(log_probs[0].item())


def rollout_branch(
    model: Model,
    env,
    prefix: list[dict],
    *,
    horizon: int = BRANCH_HORIZON,
) -> tuple[list[dict], float]:
    """Stochastic completion from a shared prefix. Returns (suffix_rows, return)."""
    context = list(prefix)
    suffix: list[dict] = []
    ret = 0.0
    model.eval()
    for _ in range(horizon):
        inputs, log_prob = act_once(model, context)
        out = env.step(inputs)
        row = {**inputs, **out, "old_log_prob": log_prob}
        row.pop("info", None)
        suffix.append(row)
        context.append(row)
        ret += float(row["reward"])
        if int(row["done"]) in (3, 4):  # task boundary — stop this branch
            break
    return suffix, ret


def fork_group(
    model: Model,
    env,
    prefix: list[dict],
    *,
    group_size: int = GROUP_SIZE,
) -> tuple[list[list[dict]], list[float]]:
    """Fork env+context into G stochastic completions.

    Returns parallel lists: per-branch full sequences (prefix + suffix) and
    per-branch scalar returns (suffix reward sum only).
    """
    branches: list[list[dict]] = []
    returns: list[float] = []
    for _ in range(group_size):
        env_g = copy.deepcopy(env)
        suffix, ret = rollout_branch(model, env_g, prefix)
        # Full sequence for training forwards; advantages stamped later.
        branches.append([dict(r) for r in prefix] + suffix)
        returns.append(ret)
        env_g.close()
    return branches, returns


def stamp_advantages_with_prefix_len(
    branches: list[list[dict]],
    returns: list[float],
    prefix_len: int,
) -> list[list[dict]]:
    adv = group_relative_advantages(torch.tensor(returns, dtype=torch.float32))
    out: list[list[dict]] = []
    for g, rows in enumerate(branches):
        a = float(adv[g].item())
        new_rows = []
        for t, row in enumerate(rows):
            r = dict(row)
            r["advantage"] = 0.0 if t < prefix_len else a
            r.setdefault("old_log_prob", 0.0)  # prefix trunk steps may lack it
            new_rows.append(r)
        out.append(new_rows)
    return out


## Collect: trunk growth + forks at many `L`

Each cycle:

1. Advance every trunk one step (stochastic), appending to its context deque.
2. When `len(context)` is a multiple of `BRANCH_EVERY` (and at least 1), fork a GRPO group at that `L`.
3. Append stamped branch sequences into on-policy `Datastore`s for training.

Trunk contexts keep growing (bounded by `MAX_CONTEXT`) so later forks see longer histories.


In [ ]:
trunk_contexts: list[deque] = [deque(maxlen=MAX_CONTEXT) for _ in trunk_envs]
# Filled each cycle with one Datastore per branch completion (cleared after train).
stores: list[Datastore] = []


def ensure_trunk_seed(model: Model, env, context: deque) -> None:
    """Take a random step if the trunk has no context yet."""
    if context:
        return
    inputs = env.sample_random_input()
    out = env.step(inputs)
    row = {**inputs, **out, "old_log_prob": 0.0, "advantage": 0.0}
    row.pop("info", None)
    context.append(row)


def collect_cycle(model: Model, grad_steps: int) -> tuple[int, int]:
    """Grow trunks, fork at selected L, fill on-policy stores. Returns (forks, branches)."""
    global stores
    model.eval()
    stores = []
    n_forks = 0
    n_branches = 0
    for step in range(TRUNK_STEPS_PER_CYCLE):
        for env, context in zip(trunk_envs, trunk_contexts):
            ensure_trunk_seed(model, env, context)
            inputs, log_prob = act_once(model, list(context))
            out = env.step(inputs)
            row = {**inputs, **out, "old_log_prob": log_prob, "advantage": 0.0}
            row.pop("info", None)
            context.append(row)

            L = len(context)
            if L > 0 and L % BRANCH_EVERY == 0:
                prefix = list(context)
                branches, returns = fork_group(model, env, prefix, group_size=GROUP_SIZE)
                stamped = stamp_advantages_with_prefix_len(branches, returns, prefix_len=L)
                for seq in stamped:
                    store = Datastore(name=f"grpo_branch_{n_branches}")
                    for r in seq:
                        store.append(r)
                    stores.append(store)
                    n_branches += 1
                n_forks += 1
                adv = group_relative_advantages(torch.tensor(returns, dtype=torch.float32))
                print(
                    f"  fork L={L} | returns={[round(r, 2) for r in returns]}"
                    f" | advantages={[round(float(a), 2) for a in adv.tolist()]}"
                    f" | grad_step={grad_steps}"
                )
        if (step + 1) % 32 == 0:
            print(f"  trunk step {step + 1}/{TRUNK_STEPS_PER_CYCLE} | forks_so_far={n_forks}")
    return n_forks, n_branches


## Train with `GrpoObjective`

Each cycle builds a fresh `DataLoader` over that cycle's branch stores. Inject both `old_log_prob` and `advantage` from the sampled rows (they are not encoder modalities). No `polyak_update` — there is no target / value net.


In [ ]:
objective = GrpoObjective(clip_eps=0.2, ent_coef=0.01, num_actions=MAX_ACTIONS)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)


def train(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: GrpoObjective,
    stores: list[Datastore],
    grad_steps: int,
    num_updates: int,
) -> None:
    """Build a loader over this cycle's branch stores and run GRPO updates."""
    if not stores or sum(len(s) for s in stores) == 0:
        print("  train skipped — on-policy stores are empty")
        return
    model.train()
    loader = DataLoader(
        stores,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=min(BATCH_SIZE, len(stores)),
        weight_mode="per_step",
        pack=True,
        num_workers=0,
    )
    try:
        for update in range(num_updates):
            batch, segment_ids = loader.next_batch()
            predictions, objective_data, _ = model(batch, segment_ids=segment_ids)
            objective_data["old_log_prob"] = batch_field(
                batch, "old_log_prob", device=objective_data.device
            )
            objective_data["advantage"] = batch_field(
                batch, "advantage", device=objective_data.device
            )
            loss, metrics = objective(objective_data, predictions)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if (update + 1) % 25 == 0 or update + 1 == num_updates:
                print(
                    f"grad_step={grad_steps + update + 1}"
                    f" | loss={loss.item():.4f}"
                    f" policy={metrics['policy_loss']:.4f}"
                    f" ent={metrics['entropy']:.4f}"
                    f" kl={metrics['approx_kl']:.4f}"
                    f" adv={metrics['advantage_mean']:.3f}"
                )
    finally:
        loader.close()


## Run

Each cycle collects forks at several context lengths, runs a few GRPO epochs, then clears the branch stores so the next cycle is on-policy again.


In [ ]:
grad_steps = 0
cycle = 0

while grad_steps < GRADIENT_STEPS:
    cycle += 1
    print(f"=== cycle {cycle} ===")
    n_forks, n_branches = collect_cycle(model, grad_steps)
    print(f"collected {n_forks} forks / {n_branches} branches")

    if n_branches == 0:
        continue

    for _ in range(GRPO_EPOCHS):
        n = min(UPDATES_PER_EPOCH, GRADIENT_STEPS - grad_steps)
        if n <= 0:
            break
        train(model, optimizer, objective, stores, grad_steps, n)
        grad_steps += n

    stores.clear()
    if device.type == "cuda":
        torch.cuda.empty_cache()

for env in trunk_envs:
    env.close()
print(f"GRPO demo finished ({grad_steps} optimizer steps, {cycle} cycles).")


## Design notes (why this shape)

| Piece | Choice in this notebook |
|-------|-------------------------|
| Group | `G = GROUP_SIZE` deepcopy'd envs + identical `context[:L]` |
| Divergence | `sample_discrete_action` (stochastic policy) |
| Where to fork | every `BRANCH_EVERY` trunk steps → many `L` |
| Score | sum of rewards on the **suffix** only |
| Advantage | `group_relative_advantages(returns)` broadcast on suffix; `0` on prefix |
| Loss | `GrpoObjective` clipped surrogate + entropy (no critic) |

**Still open / scale-up knobs:** fork at task boundaries only; longer `BRANCH_HORIZON`; KL-to-reference term; caching KV across trunk steps; forking from a `GroupEnv` of many trunks in parallel.

## Push (optional)


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
